# Tracking the Emissions of Sentiment Analysis

Measuring and benchmarking dictionary and transformer sentiment analysis
with CodeCarbon

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/seankellyhp/eco-measure-amsterdam/blob/main/notebooks/sentiment_tracking.ipynb)

In [ ]:
# Setup. On Colab: install the packages Colab does not have, and download the data.
# Locally (uv sync), everything is already installed and in the repository.
import os
import sys
import urllib.request
import warnings
from pathlib import Path

if "google.colab" in sys.modules:
    # Colab's GPU runtime has cudf-polars, which needs an older polars; these notebooks do not use it.
    %pip uninstall -q -y cudf-polars-cu13 cudf-polars-cu12
    %pip install -q -U codecarbon "ecologits[openai]" polars plotnine

# The notebooks read the data from ../data, the data folder of the repository.
for file in ["data/austen_books.csv", "data/bing.csv"]:
    path = Path("..") / file
    if not path.exists():
        path.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/seankellyhp/eco-measure-amsterdam/main/{file}", path)

# Quieten CodeCarbon before the first tracker is created.
os.environ["CODECARBON_LOG_LEVEL"] = "error"
warnings.filterwarnings("ignore")

import plotnine
import polars as pl

plotnine.options.figure_size = (7, 3.5)
plotnine.options.dpi = 110
pl.Config.set_tbl_hide_dataframe_shape(True)

## Description

This notebook continues the Getting Started tutorial. It measures two
sentiment analyses, a dictionary method and a transformer model, each with a
`with OfflineEmissionsTracker()` block, and then benchmarks three ways of
labeling sentiment against each other with CodeCarbon’s tasks.

It runs on its own: the setup below loads the packages, the data, and the
helper functions from Getting Started.

## Setup

In [ ]:
import random
import sys
import tempfile
from pathlib import Path

import nltk
import polars as pl
import torch
from codecarbon import OfflineEmissionsTracker
from plotnine import aes, geom_jitter, geom_violin, ggplot, labs, scale_y_log10, theme, theme_minimal
from transformers import pipeline

In [ ]:
austen_books = pl.read_csv("../data/austen_books.csv").with_columns(
    pl.col("text").fill_null("")
)

NLTK provides the word and sentence tokenizers and the VADER sentiment analyzer
(Bird et al., 2009). NLTK downloads them once, into one of
the folders NLTK searches:

In [ ]:
nltk.download(
    ["punkt_tab", "vader_lexicon"],
    download_dir=f"{sys.prefix}/nltk_data",
    quiet=True,
)

from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.tokenize import sent_tokenize, word_tokenize

Each tracker is created with the same settings, so keep them in one place. A
temporary folder holds the log here; use a folder of your own to keep it
after the session ends:

In [ ]:
log_dir = Path(tempfile.mkdtemp(prefix="codecarbon_results_"))

tracker_args = dict(output_dir=log_dir, country_iso_code="DEU", log_level="error")

Splitting text into words, and into sentences, is needed several times below,
so each gets a function of its own. `unnest_tokens()` is the same as in
Getting Started:

In [ ]:
def unnest_tokens(data, column):
    """Split text into words, one row each, like tidytext's unnest_tokens().

    NLTK's word tokenizer finds the words, which are then lower-cased.
    Tokens with no letter or digit in them (punctuation) are dropped, and
    so are the "'s" and "n't" that the tokenizer splits off words.
    """
    return (
        data
        .with_columns(
            pl.col(column)
            .map_elements(word_tokenize, return_dtype=pl.List(pl.String)).alias("word")
        )
        .drop(column)
        .explode("word")
        .with_columns(pl.col("word").str.to_lowercase())
        .filter(
            pl.col("word").str.contains(r"[\p{L}\p{N}]"),
            ~pl.col("word").is_in(["'s", "n't"]),
        )
    )


def unnest_sentences(data, column):
    """Split text into sentences, one row each, like tidytext's unnest_sentences().

    NLTK's sentence tokenizer finds the sentences, and `id` numbers them
    from 1 in order.
    """
    return (
        data
        .with_columns(
            pl.col(column)
            .map_elements(sent_tokenize, return_dtype=pl.List(pl.String)).alias("sentence")
        )
        .drop(column)
        .explode("sentence")
        .with_row_index("id", offset=1)
        .with_columns(
            pl.col("id")
            .cast(pl.Int64)
        )
    )

Number the chapters of each novel. A new chapter starts at each line that
begins with “Chapter”:

In [ ]:
books = austen_books.with_columns(
    linenumber=pl.int_range(1, pl.len() + 1).over("book"),
    chapter=pl.col("text").str.contains(r"(?i)^chapter [\divxlc]").cum_sum().over("book"),
)

## Comparisons of Simple Sentiment Analysis Approaches

Next we will use three approaches for sentiment analysis, with a focus on
comparing their emissions: a word count with the Bing dictionary, written
with polars; NLTK’s VADER, a dictionary with rules; and a transformer model
that has learned sentiment from labeled examples.

### Data Preparation

Sentiment is scored sentence by sentence, for every sentence of
*Pride and Prejudice*. The data has one row per printed line, so first join
each chapter’s lines back into one string. The titles in front of names are
written without their full stop, or some “Mr.” would still end a sentence:

In [ ]:
chapters = (
    books.filter(
        pl.col("book") == "Pride & Prejudice",
        pl.col("chapter") > 0,
        ~pl.col("text").str.contains(r"(?i)^chapter [\divxlc]"),
    )
    .group_by("chapter", maintain_order=True)
    .agg(text=pl.col("text").str.join(" "))
    .with_columns(
        pl.col("text")
        .str.replace_all(r"\s+", " ")
        .str.strip_chars()
        .str.replace_all(r"\b(Mr|Mrs|Dr)\.", "$1")
    )
)

Then split the chapters into sentences, one row each:

In [ ]:
pride = unnest_sentences(chapters, "text")

pride

### Dictionary Sentiment using polars

The first approach is to split sentences into words, join the Bing lexicon
(Hu & Liu, 2004), and count positive against negative words. A sentence
with no lexicon word, or as many positive as negative ones, is neutral:

In [ ]:
bing = pl.read_csv("../data/bing.csv")


def score_with_dictionary(data):
    scores = (
        unnest_tokens(data.select("id", "sentence"), "sentence")
        .join(bing, on="word")
        .group_by("id")
        .agg(score=pl.when(pl.col("sentiment") == "positive").then(1).otherwise(-1).sum())
    )
    return (
        data.select("id")
        .join(scores, on="id", how="left")
        .select(
            "id",
            sentiment=pl.when(pl.col("score") > 0)
            .then(pl.lit("positive"))
            .when(pl.col("score") < 0)
            .then(pl.lit("negative"))
            .otherwise(pl.lit("neutral")),
        )
    )

Put the call inside a `with OfflineEmissionsTracker()` block. The block can
hold any number of statements, and what they create is still there
afterwards. Here it scores every sentence in the novel:

In [ ]:
with OfflineEmissionsTracker(project_name="dictionary sentiment", **tracker_args) as tracker:
    dictionary_labels = score_with_dictionary(pride)

dict_emit = pl.DataFrame([vars(tracker.final_emissions_data)])

dict_emit.select(
    "project_name",
    "duration",
    energy_wh=pl.col("energy_consumed") * 1000,
    co2e_mg=pl.col("emissions") * 1e6,
)

Labeling all the sentences takes a fraction of a
second. The labels:

In [ ]:
dictionary_labels["sentiment"].value_counts(sort=True)

### Sentiment with a transformer model

Instead of counting words from a list, a transformer model reads each
sentence as a whole. Hugging Face’s transformers package (Wolf et al., 2020)
wraps a model, its tokenizer and the steps around them in a `pipeline()`.
The model here is DistilBERT (Sanh et al., 2019), fine-tuned to label
English sentences as positive or negative. The pipeline downloads it
(about 270 MB) the first time.

The model runs on a GPU if there is one: an NVIDIA GPU (on Colab, a GPU
runtime), or the GPU of an Apple Silicon Mac. Otherwise it runs on the CPU:

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"The model runs on: {device}")

In [ ]:
classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=device,
)

classifier(["I love this book.", "I hate this book."])

Each sentence gets a label and the model’s confidence in it. The model has
no neutral label. `score_with_transformers()` labels a list of sentences in
the same format as the dictionary. Sentences longer than the model’s limit
of 512 tokens are cut short:

In [ ]:
def score_with_transformers(sentences):
    results = classifier(list(sentences), truncation=True)
    return pl.DataFrame(
        {
            "id": range(1, len(results) + 1),
            "sentiment": [result["label"].lower() for result in results],
        }
    )

Track it in the same way, for the same sentences:

In [ ]:
with OfflineEmissionsTracker(project_name="transformer sentiment", **tracker_args) as tracker:
    transformer_labels = score_with_transformers(pride["sentence"])

transformer_emit = pl.DataFrame([vars(tracker.final_emissions_data)])

pl.concat([dict_emit, transformer_emit], how="diagonal_relaxed").select(
    "project_name",
    "duration",
    energy_wh=pl.col("energy_consumed") * 1000,
    co2e_mg=pl.col("emissions") * 1e6,
)

The model takes far longer, and uses far more energy, than the dictionary.
Its labels, against the dictionary’s:

In [ ]:
pl.DataFrame(
    {"dictionary": dictionary_labels["sentiment"], "transformers": transformer_labels["sentiment"]}
).pivot(
    on="transformers", index="dictionary", values="transformers", aggregate_function="len", sort_columns=True
).sort("dictionary")

The model calls every sentence positive or negative, including the ones
the dictionary leaves neutral, and the two disagree on many of the others.
Neither is the truth: to know which is closer to it, compare both with
sentences labeled by hand.

### Benchmark: polars, NLTK or transformers?

NLTK (Bird et al., 2009) has a sentiment analyzer of its own, VADER
(Hutto & Gilbert, 2014). It is also built on a dictionary, but each word in it
has a strength, not just a sign, and rules adjust them: for negation (“not
good”), for words like “very”, for capital letters and for exclamation
marks. `polarity_scores()` sums them into a `compound` score from -1 to 1:

In [ ]:
sia = SentimentIntensityAnalyzer()

sia.polarity_scores("I love this product! It's amazing.")

VADER’s authors call a sentence positive when its compound score is at least
0.05, negative when it is at most -0.05, and neutral in between:

In [ ]:
def get_vader_sentiment(text):
    compound = sia.polarity_scores(text)["compound"]
    if compound >= 0.05:
        return "positive"
    elif compound <= -0.05:
        return "negative"
    else:
        return "neutral"


get_vader_sentiment("I love this product! It's amazing.")

To compare the three approaches fairly, all start from the same texts from
*Pride and Prejudice*, one row per chapter. All then split the chapters into
sentences and label every sentence.

In [ ]:
def sentiment_polars(texts):
    return score_with_dictionary(unnest_sentences(texts, "text"))


def score_with_nltk(sentences):
    labels = [get_vader_sentiment(sentence) for sentence in sentences]
    return pl.DataFrame({"id": range(1, len(labels) + 1), "sentiment": labels})


def sentiment_nltk(texts):
    sentences = [sentence for text in texts["text"] for sentence in sent_tokenize(text)]
    return score_with_nltk(sentences)


def sentiment_transformers(texts):
    sentences = [sentence for text in texts["text"] for sentence in sent_tokenize(text)]
    return score_with_transformers(sentences)

All three split the chapters with NLTK’s sentence tokenizer; only the way they
label each sentence differs. To compare the two dictionaries’ labels, score
the same sentences, `pride`, with VADER. The confusion matrix shows that
they agree on about two sentences in three. VADER finds sentiment in many
sentences the Bing count leaves neutral, and it more often calls a sentence
positive:

In [ ]:
nltk_labels = score_with_nltk(pride["sentence"])

pl.DataFrame(
    {"polars": dictionary_labels["sentiment"], "nltk": nltk_labels["sentiment"]}
).pivot(on="nltk", index="polars", values="nltk", aggregate_function="len", sort_columns=True).sort("polars")

CodeCarbon has no benchmark of its own, but tasks make one. `carbon_bench()`
runs each function repeatedly, in random order, as one task each, after one
untracked warm-up run of each:

In [ ]:
def carbon_bench(functions, times=10, warmup=True):
    if warmup:
        for function in functions.values():
            function()

    runs = [(name, i) for name in functions for i in range(1, times + 1)]
    random.shuffle(runs)

    tracker = OfflineEmissionsTracker(project_name="carbon bench", **tracker_args)
    rows = []
    try:
        for name, i in runs:
            tracker.start_task(f"{name}_{i}")
            functions[name]()
            task = tracker.stop_task()
            rows.append({"expr": name, "iter": i, **vars(task)})
    finally:
        tracker.stop()

    return pl.DataFrame(rows)

`plot_bench()` draws the runs of each function, one point per run over the
shape of their distribution:

In [ ]:
def plot_bench(bench, metric="energy_consumed"):
    return (
        ggplot(bench, aes("expr", metric, fill="expr"))
        + geom_violin(alpha=0.4)
        + geom_jitter(width=0.1, height=0, alpha=0.7)
        + labs(x="", y=metric, title=f"carbon_bench: {metric}")
        + theme_minimal()
        + theme(legend_position="none")
    )

Compare the three, 5 times each. The transformer model takes up most of
the time: on a CPU, about a minute per run on a fast desktop, and more on a
laptop or on Colab, so the benchmark takes ten minutes or more. A GPU speeds
it up; on Colab, choose one under *Runtime \> Change runtime type*:

In [ ]:
random.seed(1234)

bench = carbon_bench(
    {
        "polars": lambda: sentiment_polars(chapters),
        "nltk": lambda: sentiment_nltk(chapters),
        "transformers": lambda: sentiment_transformers(chapters),
    },
    times=5,
)

bench.group_by("expr", maintain_order=True).agg(
    duration_sec=pl.col("duration").median(),
    energy_wh=pl.col("energy_consumed").median() * 1000,
    co2e_mg=pl.col("emissions").median() * 1e6,
)

In [ ]:
plot_bench(bench, metric="energy_consumed")

The transformer model uses orders of magnitude more energy than either
dictionary, so the two dictionaries are squeezed together at the bottom.
A log scale shows all three:

In [ ]:
plot_bench(bench, metric="energy_consumed") + scale_y_log10()

## Technical Details

See the [CodeCarbon documentation](https://mlco2.github.io/codecarbon/) and
the [EcoLogits documentation](https://ecologits.ai) for their methods, and
the [TidyCarbon repository](https://github.com/seankellyhp/Tidycarbon) for
the R version of this tutorial.

## References

Bird, S., Klein, E., & Loper, E. (2009). *Natural language processing with Python: Analyzing text with the natural language toolkit*. O’Reilly Media.

Hu, M., & Liu, B. (2004). Mining and summarizing customer reviews. *Proceedings of the Tenth ACM SIGKDD International Conference on Knowledge Discovery and Data Mining*, 168–177.

Hutto, C. J., & Gilbert, E. (2014). VADER: A parsimonious rule-based model for sentiment analysis of social media text. *Proceedings of the Eighth International AAAI Conference on Weblogs and Social Media*, 216–225. <https://doi.org/10.1609/icwsm.v8i1.14550>

Sanh, V., Debut, L., Chaumond, J., & Wolf, T. (2019). *DistilBERT, a distilled version of BERT: Smaller, faster, cheaper and lighter*. <https://doi.org/10.48550/arXiv.1910.01108>

Wolf, T., Debut, L., Sanh, V., Chaumond, J., Delangue, C., Moi, A., Cistac, P., Rault, T., Louf, R., Funtowicz, M., Davison, J., Shleifer, S., Platen, P. von, Ma, C., Jernite, Y., Plu, J., Xu, C., Le Scao, T., Gugger, S., … Rush, A. M. (2020). Transformers: State-of-the-art natural language processing. *Proceedings of the 2020 Conference on Empirical Methods in Natural Language Processing: System Demonstrations*, 38–45. <https://doi.org/10.18653/v1/2020.emnlp-demos.6>

## Disclaimer

This method is intended for academic research; ensure legality and ethics
when processing chat data.

## Contact Details